# APS360 super-resolution: Colab runner
Runtime -> Change runtime type -> **T4 GPU**. Run top to bottom. Checkpoints go to Google Drive so a disconnect doesn't lose a run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/aps360_sr_runs'
!mkdir -p $RUNS
!nvidia-smi -L

## 1. Code
Replace the URL with your repo (or upload the zip and unzip it to /content/sr-project).

In [ ]:
REPO = 'https://github.com/YOUR_USERNAME/aps360-super-resolution.git'
!git clone -q $REPO /content/sr-project
%cd /content/sr-project
!pip -q install -r requirements.txt

## 2. Data (~5 min)

In [ ]:
!mkdir -p data
!wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip -P data && unzip -q data/DIV2K_train_HR.zip -d data
!wget -q http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip -P data && unzip -q data/DIV2K_valid_HR.zip -d data
!wget -q https://cv.snu.ac.kr/research/EDSR/benchmark.tar -P data && tar -xf data/benchmark.tar -C data
!ls data data/benchmark

In [ ]:
!python prepare_data.py --src data/DIV2K_train_HR --dst data/div2k_train_npy
!python prepare_data.py --src data/DIV2K_valid_HR --dst data/div2k_valid_npy

## 3. Smoke test + leakage check

In [ ]:
!python smoke_test.py

In [ ]:
!python check_data.py --train data/DIV2K_train_HR data/DIV2K_valid_HR --bench data/benchmark/Set5 data/benchmark/Set14 data/benchmark/B100

## 4. Fixed test sets + bicubic baseline
Run `make_testsets.py` once; copy `testsets/degraded_params.csv` and `testsets/MANIFEST.txt` into your repo.

In [ ]:
!python make_testsets.py --bench_root data/benchmark --sets Set5 Set14 B100
!cp -r testsets $RUNS/
!python evaluate.py --methods bicubic
!python count_params.py

## 5. Train (each ~30-45 min on a T4)
If you are short on time, lower `--iters` (15000 still beats bicubic). Rerun the same cell with `--resume` after a disconnect.

In [ ]:
!python train.py --name clean --model edsr --iters 30000 --runs $RUNS

In [ ]:
!python train.py --name degraded --model edsr --degraded --iters 30000 --runs $RUNS

## 6. Results for the report

In [ ]:
!python evaluate.py --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt
!python plot_curves.py $RUNS/clean $RUNS/degraded
!python visualize.py --set Set14 --image comic.png --condition clean --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt
!python visualize.py --set B100 --image 253027.png --condition b1.0_n5_q75 --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt
!cp -r results $RUNS/

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open('results/table.md').read()))
import glob
for f in sorted(glob.glob('results/figures/*.png')): display(Image(f))

## Later (after the progress report)

In [ ]:
# !python train.py --name degaware --model degaware --degraded --iters 30000 --runs $RUNS
# !python train.py --name srcnn --model srcnn --iters 30000 --runs $RUNS
# !python visualize.py --phone_dir data/phone --methods bicubic $RUNS/clean/best.pt $RUNS/degraded/best.pt $RUNS/degaware/best.pt